# 第 7 章 · 多 agent（Parallel Processing）

两个 agent **同时**跑，各写各的 state 键，最后合起来看。

和 Sequential 的唯一区别是并发：它们之间不能有数据依赖。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, quiet_adk_sync_warnings, retry_call
quiet_adk_sync_warnings()

from google.adk.runners import InMemoryRunner
from google.genai import types


def run_once(agent, message):
    """跑一次，返回 (每个 agent 的最终答复, 结束时 session.state)。"""
    runner = InMemoryRunner(agent)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    finals = []
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=message)]))), label="run "):
        if ev.is_final_response() and ev.content:
            t = getattr(ev.content, "text", None) or "".join(
                p.text for p in (ev.content.parts or []) if getattr(p, "text", None))
            finals.append((getattr(ev, "author", "?"), (t or "").strip()))
    sess = runner.session_service.get_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    return finals, sess.state

from google.adk.agents import LlmAgent, ParallelAgent

weather_fetcher = LlmAgent(
    name="weather_fetcher",
    model=adk_model(),                      # 【适配】gemini-2.0-flash-exp 已下线
    instruction="给出所给地点的天气，只回一句话。",
    output_key="weather_data",
)

news_fetcher = LlmAgent(
    name="news_fetcher",
    model=adk_model(),                      # 【适配】同上
    instruction="给出所给主题的一条新闻标题和一句话摘要。",
    output_key="news_data",
)

# ★ 本章新增 ParallelAgent：子 agent 同时启动，全部结束后才算自己结束
data_gatherer = ParallelAgent(name="data_gatherer", sub_agents=[weather_fetcher, news_fetcher])

finals, state = run_once(data_gatherer, "地点：北京；主题：人工智能")
for author, text in finals:
    print(f"\n--- [{author}] ---\n{text[:300]}")
print("\n--- 汇合后的 state ---")
print("weather_data:", (state.get("weather_data") or "(空)")[:200])
print("news_data   :", (state.get("news_data") or "(空)")[:200])


--- [news_fetcher] ---
**新闻标题：**
北京发布首批人工智能大模型应用场景，加速AI赋能千行百业

**一句话摘要：**
北京市近日正式发布首批人工智能大模型应用示范场景，涵盖智慧城市、医疗健康、工业制造等多个领域，旨在通过政策引导与场景开放，加速AI技术在实体经济中的深度融合与落地应用。

--- [weather_fetcher] ---
根据人工智能气象数据预测，今日北京晴朗少云，气温15℃至25℃，非常适合出行。

--- 汇合后的 state ---
weather_data: 根据人工智能气象数据预测，今日北京晴朗少云，气温15℃至25℃，非常适合出行。
news_data   : **新闻标题：**
北京发布首批人工智能大模型应用场景，加速AI赋能千行百业

**一句话摘要：**
北京市近日正式发布首批人工智能大模型应用示范场景，涵盖智慧城市、医疗健康、工业制造等多个领域，旨在通过政策引导与场景开放，加速AI技术在实体经济中的深度融合与落地应用。
